# DefectosCafeVerde — DVF1 three-seed confirmation decision

Aggregates fixed seeds 42, 123, and 2026. No training and no test access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json

roots=[Path('/content/drive/MyDrive'),Path('/content/drive/.shortcut-targets-by-id')]
relative=Path('experiments/defectoscafeverde-dvf1-v1/DVF1_seed42_result.json')
matches=sorted({p for root in roots if root.exists() for p in root.rglob(relative.name) if p.as_posix().endswith(relative.as_posix())})
if not matches: raise FileNotFoundError(relative)
preferred=[p for p in matches if '/MyDrive/' in p.as_posix()]
SEED42=(preferred or matches)[0]; PROJECT=SEED42.parents[2]
CONFIRM=PROJECT/'experiments/defectoscafeverde-dvf1-confirmation-v1'
SEED123=CONFIRM/'DVF1_seed123_result.json'; SEED2026=CONFIRM/'DVF1_seed2026_result.json'
for path in (SEED42,SEED123,SEED2026):
    if not path.is_file(): raise FileNotFoundError(path)
SUMMARY=CONFIRM/'DVF1_three_seed_confirmation.json'
print('PROJECT:',PROJECT); print('SUMMARY:',SUMMARY)


In [ ]:
import importlib, os, shutil, subprocess, sys
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'; BRANCH='codex/defectoscafeverde-dual-view-fusion'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_defectoscafeverde_dvf1_confirmation_decision','--seed42-result',str(SEED42),'--seed123-result',str(SEED123),'--seed2026-result',str(SEED2026),'--output',str(SUMMARY)]
subprocess.run(command,cwd=REPO,check=True)
result=json.loads(SUMMARY.read_text()); assert result['training_executed'] is False and result['test_images_accessed'] is False


In [ ]:
import pandas as pd
rows=[]
for metric,row in result['aggregate'].items(): rows.append({'metric':metric,**row})
display(pd.DataFrame(rows).style.format({'paper_mean':'{:.2%}','dvf1_mean':'{:.2%}','dvf1_std':'{:.2%}','delta_mean':'{:+.2%}','delta_min':'{:+.2%}'}))
print('PER SEED:',json.dumps(result['per_seed'],indent=2))
print('CRITERIA:',result['criteria'])
print('DECISION:',result['decision'],'| NEXT:',result['next'])
print('TRAINING:',result['training_executed'],'| TEST:',result['test_images_accessed'])
